# Data Quality Audit — 02 · Riyadh Places 8.8K (Kaggle)

**Source:** `data/raw/riyadh_places/riyadh_places_8836x9.csv`

**Purpose in the concierge:** POIs / restaurants with categories, ratings, and coordinates.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)


repo root: /home/user/saudi-Digital-Concierge


In [2]:
df = pd.read_csv(ROOT / "data/raw/riyadh_places/riyadh_places_8836x9.csv")
print("Loaded places:", df.shape)
df.head(3)

Loaded places: (8836, 9)


,id,place_name,is_restaurant,categories,average_rating,rate_count,granular_category,latitude,longitude
0,1,Shawarma Alaz,RESTAURANT,Sandwich|Fast Food|Shawarma,3.0,3,restaurants,24.645118,46.717640
1,2,Eben Ezar,RESTAURANT,Asian|Japanese,3.2,5,restaurants,24.646631,46.717658
2,3,Chowking,RESTAURANT,Noodles,4.5,305,restaurants,24.646194,46.717444


## Shape

In [3]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 8836 | Columns: 9


## Columns & data types

In [4]:
dtype_report(df)

,dtype,non_null,n_unique
id,int64,8836,8836
place_name,str,8836,5249
is_restaurant,str,8836,2
categories,str,8822,1046
average_rating,float64,8836,39
rate_count,int64,8836,1019
granular_category,str,8836,25
latitude,float64,8836,7188
longitude,float64,8836,7184


## Missing values

In [5]:
missing_report(df) if not missing_report(df).empty else print('No missing values')

,missing,missing_%
categories,14,0.2


## Duplicates

In [6]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate id:", df["id"].duplicated().sum())
print("Duplicate (name, lat, lon):", df.duplicated(subset=["place_name","latitude","longitude"]).sum())

Exact duplicate rows: 0
Duplicate id: 0
Duplicate (name, lat, lon): 0


## Invalid values
`average_rating` should be 0–5; `rate_count` should be ≥ 0.

In [7]:
r = pd.to_numeric(df["average_rating"], errors="coerce")
print("rating out of 0-5:", ((r < 0) | (r > 5)).sum())
print("rating == 0 (unrated?):", (r == 0).sum())
rc = pd.to_numeric(df["rate_count"], errors="coerce")
print("negative rate_count:", (rc < 0).sum())
print("is_restaurant values:", df["is_restaurant"].value_counts(dropna=False).to_dict())

rating out of 0-5: 0
rating == 0 (unrated?): 749
negative rate_count: 0
is_restaurant values: {'RESTAURANT': 8703, 'OTHER': 133}


## Outliers
`rate_count` is heavily skewed; report IQR outliers.

In [8]:
n, lo, hi = iqr_outliers(df["rate_count"])
print(f"rate_count IQR outliers: {n} (bounds {lo}..{hi})")
print(df["rate_count"].describe().round(1).to_string())

rate_count IQR outliers: 988 (bounds -217.0..383.0)
count     8836.0
mean       183.6
std        589.1
min          0.0
25%          8.0
50%         39.0
75%        158.0
max      19808.0


## Inconsistent categories
`categories` is pipe-delimited (multi-value); `granular_category` is a single label.

In [9]:
print("granular_category:")
print(df["granular_category"].value_counts(dropna=False).head(20).to_string())
print("\nTop individual categories (exploded):")
print(df["categories"].dropna().str.split("|").explode().str.strip().value_counts().head(20).to_string())

granular_category:
granular_category
restaurants              8487
flowers_and_plants        143
pharmacies                 65
Butchery                   18
specialty_and_ethnic       16
charity                    15
supermarket                12
darkstores                 11
Hypermarket                11
nuts_and_dried_fruits       7
Pets                        7
fruits_and_vegetables       6
beauty                      5
dairy_products              5
electronics                 5
health_and_wellness         4
home_and_gifts              4
mini_market                 3
optics                      3
bakery                      2

Top individual categories (exploded):
categories
Desserts         2696
Arabic           2486
Fast Food        1854
Sandwich         1473
Coffee           1182
Beverages        1131
Burgers           836
American          740
Breakfast         640
Pizza             593
Bakery            573
Grill             530
Saudi             438
Italian           421
Shawa

## Geographic validity
All points should fall inside the Saudi bounding box (and, being Riyadh, cluster tightly).

In [10]:
lat = pd.to_numeric(df["latitude"], errors="coerce")
lon = pd.to_numeric(df["longitude"], errors="coerce")
in_sa = lat.between(*SA_LAT) & lon.between(*SA_LON)
print("Missing coords:", lat.isna().sum())
print("Outside Saudi bbox:", (~in_sa).sum())
print("Lat range:", round(lat.min(),3), "-", round(lat.max(),3))
print("Lon range:", round(lon.min(),3), "-", round(lon.max(),3))

Missing coords: 0
Outside Saudi bbox: 0
Lat range: 24.488 - 26.445
Lon range: 46.517 - 50.131


## Date/time validity
No date/time columns — **N/A**.

In [11]:
print("Date-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["date","time","year"])])

Date-like columns: []


## Data-source / license
- **Source:** Kaggle — *Riyadh Places 8.8K*.
- **License:** TBD (confirm on the Kaggle page).
- **Currency:** static snapshot.

## Known limitations
- **Riyadh only** — no national coverage.
- `average_rating == 0` likely means *unrated*, not a 0-star place.
- `categories` is multi-valued (pipe-delimited) → explode/normalize for filtering.
- License to confirm.